# Fuzzy Membership Finetuning

Fine-tunes a small BERT model on soft (fuzzy) cluster membership vectors using
the `fuzzyfinetuning_crossentropy.py` script from the RecsysUpgrade repo.

Reads the membership matrix saved by the clustering notebook (`fkmeans_memberships.csv`
and `fkmeans_clusters.csv`) from the same `CLUSTERS_OUTPUT_DIR`.

### Step 1: Setup (drive mounting, paths)

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# ── Match these paths to clustering_comparison.ipynb ──────────────────────────
DRIVE_BASE          = "/content/drive/MyDrive/S2026/Spotify Playlist Data/Processed Data/calced_embeddings"
PROJECT_BASE        = "/content/drive/MyDrive/S2026/CS 274/Playlist-Recommender"
EMBEDDINGS_PKL      = f"{DRIVE_BASE}/mean_pooled_embeddings.pkl"
CLUSTERS_OUTPUT_DIR = f"{PROJECT_BASE}/new clusters"
# ──────────────────────────────────────────────────────────────────────────────

FINETUNED_MODEL_DIR = f"{PROJECT_BASE}/fuzzy_finetuned_model"
REPO_DIR            = "/content/RecsysUpgrade"

os.makedirs(FINETUNED_MODEL_DIR, exist_ok=True)
print("Paths configured.")

### Step 2: Clone repo and install requirements

In [ ]:
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/siddmohanty111/RecsysUpgrade.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

print("Repo ready:", REPO_DIR)

In [ ]:
%pip install -r {REPO_DIR}/requirements.txt -q

### Step 3: Load membership matrix and playlist titles

In [ ]:
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# fkmeans_clusters.csv was saved alongside fkmeans_memberships.csv from the same `pids` list,
# so row i in memberships corresponds to row i (pid) in clusters.
clusters_df = pd.read_csv(
    os.path.join(CLUSTERS_OUTPUT_DIR, "fkmeans_clusters.csv"),
    dtype={"pid": str},
)
pids_ordered = clusters_df["pid"].tolist()

memberships_df = pd.read_csv(os.path.join(CLUSTERS_OUTPUT_DIR, "fkmeans_memberships.csv"))
membership_cols = [c for c in memberships_df.columns if c.startswith("cluster_")]
num_clusters = len(membership_cols)
print(f"Membership matrix: {len(pids_ordered)} playlists × {num_clusters} clusters")

# Load playlist titles from the embeddings pkl (same source as the clustering notebook)
with open(EMBEDDINGS_PKL, "rb") as f:
    raw_data = pickle.load(f)

playlist_titles = (
    raw_data.get("playlist_titles", {})
    if isinstance(raw_data, dict)
    else {}
)
print(f"Playlist titles available: {len(playlist_titles)}")

### Step 4: Build dataset and train/val split

In [ ]:
# Build a DataFrame with the two columns expected by fuzzyfinetuning_crossentropy.run():
#   "Playlist Title"  — string title
#   "Cluster Labels"  — string repr of a Python list, e.g. "[0.12, 0.03, ...]"
#                       (parsed by ast.literal_eval inside the finetuning script)

data_df = pd.DataFrame({
    "Playlist Title": [playlist_titles.get(pid, "") for pid in pids_ordered],
    "Cluster Labels": memberships_df[membership_cols].values.tolist(),
})

# Convert each membership list to its string representation
data_df["Cluster Labels"] = data_df["Cluster Labels"].apply(str)

# Drop playlists with no title — they carry no signal for the model
data_df = data_df[data_df["Playlist Title"].str.strip() != ""].reset_index(drop=True)
print(f"Dataset after dropping untitled playlists: {len(data_df)}")

# 90 / 10 train–val split
train_df, val_df = train_test_split(data_df, test_size=0.1, random_state=42)

TRAIN_CSV = os.path.join(CLUSTERS_OUTPUT_DIR, "fuzzy_train.csv")
VAL_CSV   = os.path.join(CLUSTERS_OUTPUT_DIR, "fuzzy_val.csv")

train_df.to_csv(TRAIN_CSV, index=False)
val_df.to_csv(VAL_CSV,   index=False)
print(f"Train: {len(train_df)}  Val: {len(val_df)}")
print(f"Saved to {CLUSTERS_OUTPUT_DIR}")

### Step 5: Load finetuning module and run

In [ ]:
import importlib.util

ft_path = os.path.join(
    REPO_DIR, "PlaylistRecsysUpgrade", "finetuning", "fuzzyfinetuning_crossentropy.py"
)
spec   = importlib.util.spec_from_file_location("fuzzyfinetuning_crossentropy", ft_path)
fuzzy_ft = importlib.util.module_from_spec(spec)
spec.loader.exec_module(fuzzy_ft)
print("fuzzyfinetuning_crossentropy loaded.")

In [ ]:
# prajjwal1/bert-tiny  — 4.4 M params, trains ~10× faster than MiniLM on Colab.
# Swap to "prajjwal1/bert-mini" or "distilbert-base-uncased" for better quality.
fuzzy_ft.run(
    train_csv=TRAIN_CSV,
    val_csv=VAL_CSV,
    output_dir=FINETUNED_MODEL_DIR,
    model_name="prajjwal1/bert-tiny",
    batch_size=128,
    epochs=20,
    learning_rate=3e-4,
    warmup_steps=200,
)